In [2]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_capability(0))

a=torch.randn(2048,2048,device='cuda')

print((a@a).sum().item())

2.13.0+cu132
True
NVIDIA GeForce RTX 5060 Ti
(12, 0)
-129007.6875


In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct", device_map="auto")
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=256)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

d:\Repository\envs\llm_sft\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Jason\.cache\huggingface\hub\models--Qwen--Qwen2.5-1.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

I am Qwen, an AI language model developed by Alibaba Cloud. I can answer questions, engage in conversation, provide information on various topics, and assist with tasks such as writing essays or emails. Please let me know how I can help you today!<|im_end|>


In [4]:
from datasets import load_dataset

gsm_train = load_dataset("openai/gsm8k", "main", split="train")
gsm_test = load_dataset("openai/gsm8k", "main", split="test")

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

d:\Repository\envs\llm_sft\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Jason\.cache\huggingface\hub\datasets--openai--gsm8k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

In [7]:
!python ../src/build_data.py --smoke

[data] loading gsm8k ...
[data] loading hendrycks_math ...
[data] +20 MATH train prompts
[data] loading MATH-500 ...
[data] train=70 dev=10 gsm8k_test=40 math500=40
[data] example target:
---
He streams for 7-3=4 days a week
So he streams 4*4=16 hours a week
That means he makes 16*10=$160 a week

The answer is \boxed{160}.
---


In [6]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(r"D:\Repository\Jianyu-is-learning\posttrain-paper")

SRC_DIR = PROJECT_ROOT / "src"

sys.path.insert(0, str(SRC_DIR))

from build_data import gsm8k_to_target

print(gsm8k_to_target(
    "He streams for 7-3=<<7-3=4>>4 days a week\n"
    "So he streams 4*4=<<4*4=16>>16 hours a week\n"
    "That means he makes 16*10=$<<16*10=160>>160 a week\n"
    "#### 160")[1])

He streams for 7-3=4 days a week
So he streams 4*4=16 hours a week
That means he makes 16*10=$160 a week

The answer is \boxed{160}.
